# 🚀 Gemini Blog & Email Generator (Google Colab)

This notebook implements a complete content generation application using the **Google Gemini API**.

### 📋 Features & Requirements:
- **`generate_blog(topic, tone, word_count)`** - Uses system instruction `"You are an experienced blog writer..."` with `temperature = 0.7`.
- **`generate_email(recipient, purpose, tone)`** - Uses system instruction `"You are a professional email writer..."` with `temperature = 0.3`.
- **`main()`** - Interactive menu tying components together.
- **Colab Secrets Integration** - Fetches `GEMINI_API_KEY` safely using `userdata.get('GEMINI_API_KEY')`.
- **File Export** - Exports generated content to `blog_output.txt` and `email_output.txt` in the exact required format.

In [ ]:
# Step 1: Install Google GenAI SDK
!pip install -q google-genai google-generativeai

In [ ]:
# Step 2: Setup API Key retrieval from Colab Secrets
import os
import getpass

def get_api_key():
    """
    Retrieve Gemini API key from Colab Secrets (userdata), environment variables,
    or interactive prompt.
    """
    try:
        import importlib
        colab_userdata = importlib.import_module('google.colab.userdata')
        key = colab_userdata.get('GEMINI_API_KEY') or colab_userdata.get('GOOGLE_API_KEY')
        if key:
            return key
    except Exception:
        pass

    key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
    if key:
        return key

    print("⚠️ GEMINI_API_KEY not found in Colab Secrets.")
    return getpass.getpass("Please enter your Gemini API Key: ").strip()

def call_gemini(system_instruction: str, prompt: str, temperature: float) -> str:
    """
    Calls Google Gemini API with system instructions and custom temperature.
    """
    api_key = get_api_key()
    if not api_key:
        raise ValueError("GEMINI_API_KEY is required.")

    try:
        from google import genai
        from google.genai import types

        client = genai.Client(api_key=api_key)
        config = types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
        )
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt,
            config=config
        )
        return response.text.strip()
    except Exception as err_genai:
        import google.generativeai as genai_legacy
        genai_legacy.configure(api_key=api_key)
        model = genai_legacy.GenerativeModel(
            model_name="gemini-1.5-flash",
            system_instruction=system_instruction
        )
        response = model.generate_content(
            prompt,
            generation_config=genai_legacy.GenerationConfig(temperature=temperature)
        )
        return response.text.strip()

In [ ]:
# Step 3: Define generate_blog function (Temperature: 0.7)
def generate_blog(topic: str, tone: str, word_count: int, save_file: str = "blog_output.txt") -> str:
    """
    Generates a blog post using Gemini API with temperature 0.7.
    """
    system_instruction = "You are an experienced blog writer who crafts engaging, insightful, and well-structured articles."

    prompt = f"""
Write a blog post based on the following details:
- Topic: "{topic}"
- Tone: {tone}
- Target Word Count: {word_count} words

Formatting & Structure Guidelines:
1. Title: Create an engaging, catchy title.
2. Intro Hook: Start with a strong hook that captures reader interest.
3. Body Sections: Write 2 to 3 distinct body sections with descriptive subheadings.
4. Conclusion: Summarize main points with a memorable takeaway or call to action.
    """

    blog_text = call_gemini(system_instruction=system_instruction, prompt=prompt, temperature=0.7)

    header = f"===== BLOG GENERATOR =====\nTopic: \"{topic}\"\nTone: {tone}\nWord count: {word_count}\n\n"
    full_output = header + blog_text

    if save_file:
        with open(save_file, "w", encoding="utf-8") as f:
            f.write(full_output)
        print(f"💾 Saved blog output to '{save_file}'")

    return full_output

In [ ]:
# Step 4: Define generate_email function (Temperature: 0.3)
def generate_email(recipient: str, purpose: str, tone: str, save_file: str = "email_output.txt") -> str:
    """
    Generates a professional email using Gemini API with temperature 0.3.
    """
    system_instruction = "You are a professional email writer who produces clear, polished, and effective emails."

    prompt = f"""
Write an email based on the following details:
- Recipient: {recipient}
- Purpose: {purpose}
- Tone: {tone}

Formatting & Structure Guidelines:
1. Subject Line: Clear and professional subject line.
2. Salutation: Appropriate greeting to {recipient}.
3. Email Body: Concise, context-appropriate message addressing the purpose ({purpose}).
4. Closing: Professional sign-off with placeholders for sender details.
    """

    email_text = call_gemini(system_instruction=system_instruction, prompt=prompt, temperature=0.3)

    header = f"===== EMAIL GENERATOR =====\nRecipient: {recipient}\nPurpose: {purpose}\nTone: {tone}\n\n"
    full_output = header + email_text

    if save_file:
        with open(save_file, "w", encoding="utf-8") as f:
            f.write(full_output)
        print(f"💾 Saved email output to '{save_file}'")

    return full_output

In [ ]:
# Step 5: Define main() interactive menu
def main():
    """
    Interactive menu system.
    """
    print("==============================================")
    print("      GEMINI BLOG & EMAIL GENERATOR")
    print("==============================================")
    print("\nSelect an option:")
    print("1. Generate Blog Post")
    print("2. Generate Email")
    print("3. Run Demo (Sample Blog + Email)")

    choice = input("Enter choice (1-3) [default: 3]: ").strip() or "3"

    if choice == "1":
        topic = input("Topic: ").strip() or "Why Python is the best first language"
        tone = input("Tone: ").strip() or "casual"
        wc = int(input("Word Count: ").strip() or 300)
        print("\nGenerating Blog...")
        print(generate_blog(topic, tone, wc))
    elif choice == "2":
        recipient = input("Recipient: ").strip() or "HR Manager, TCS"
        purpose = input("Purpose: ").strip() or "Follow-up after interview"
        tone = input("Tone: ").strip() or "professional"
        print("\nGenerating Email...")
        print(generate_email(recipient, purpose, tone))
    else:
        print("\n--- RUNNING DEMO ---")
        print(generate_blog("Why Python is the best first language", "casual", 300))
        print("\n" + "-"*40 + "\n")
        print(generate_email("HR Manager, TCS", "Follow-up after interview", "professional"))

# Run main menu
main()